---
<div style="display: flex; justify-content: space-between; align-items: center; width: 100%;">
    <img src='../misc/ufpr.png' style='width: 15%;'>
    <div style="text-align: center; flex-grow: 1;">
        <h2 style='line-height: 0.5; color:#ffda77; margin-top: 50px;'>Universidade Federal do Paraná</h2>
        <h2 style='line-height: 0.5; color:#ffda77;'>LABAP</h2>
        <h6 style='line-height: 0.5; color:#ffda77;'>Laboratório de Análise de Bacias e Petrofísica</h6>
    </div>
    <img src='../misc/labap.png' style='width: 15%;'>
</div>
<br>

---

# 11. Sensitivity Analysis — XGBoost

## Objective

Investigate how XGBoost hyperparameters affect the prediction performance
of the sonic log (DT) in the Sergipe-Alagoas Basin.

**Key questions:**
1. Which hyperparameters have the greatest impact on LOWO R²?
2. What is the direction of the effect of each hyperparameter?
3. Are there relevant interactions between the most important parameters?
4. Is the model stable with respect to the choice of hyperparameters?

**Data:** results from `RandomizedSearchCV` (**50 trials**) salvo em `cv_results.csv`
e melhores parâmetros em `xgboost_best_params.json`.

**Optimized hyperparameters (9):** `learning_rate`, `n_estimators`, `max_depth`,
`min_child_weight`, `subsample`, `colsample_bytree`, `reg_alpha`, `reg_lambda`, `gamma`.


## 1. Imports and Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import spearmanr

import warnings
warnings.filterwarnings('ignore')

RESULTS_DIR = Path('../results/xgboost')
METRICS_DIR = RESULTS_DIR / 'metrics'
MODELS_DIR  = RESULTS_DIR / 'models'
FIGURES_DIR = RESULTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load Tuning Results

The file `cv_results.csv` contains the **50 trials** from `RandomizedSearchCV`.
Each row is a trial with the tested hyperparameters and the resulting mean R².

**Tuning results:**

| Métric | Valor |
|---|---|
| Evaluated trials | 50 |
| Best R² (GroupKFold) | **0.8900** |
| Worst R² (GroupKFold) | 0.8228 |
| Total range | 0.0671 |
| Mean ± Std Dev | 0.8733 ± 0.0190 |

**Best hyperparameters found:**

| Hyperparameter | Valor |
|---|---|
| `max_depth` | 4 |
| `learning_rate` | 0.0617 |
| `n_estimators` | 430 |
| `subsample` | 0.421 |
| `colsample_bytree` | 0.812 |
| `min_child_weight` | 2 |
| `gamma` | 3.956 |
| `reg_alpha` | 4.243 |
| `reg_lambda` | 197.67 |


In [ ]:
# cv_results: 50 trials from RandomizedSearchCV
cv_results = pd.read_csv(MODELS_DIR / 'cv_results.csv')

# Best parameters
with open(MODELS_DIR / 'xgboost_best_params.json') as f:
    best_params_data = json.load(f)

best_params   = best_params_data['best_params']
best_cv_score = best_params_data['best_cv_score']

print('RANDOMIZEDSEARCHCV RESULTS — XGBoost')
print(f'Trials              : {len(cv_results)}')
print(f'Best R²  (KFold)    : {cv_results["mean_test_score"].max():.4f}')
print(f'Worst R² (KFold)    : {cv_results["mean_test_score"].min():.4f}')
print(f'Total range         : {cv_results["mean_test_score"].max() - cv_results["mean_test_score"].min():.4f}')
print(f'Mean ± Std Dev      : {cv_results["mean_test_score"].mean():.4f} ± {cv_results["mean_test_score"].std():.4f}')
print()
print('Best hyperparameters:')
for k, v in best_params.items():
    print(f'  {k:25s}: {v}')

# Automatically detect hyperparameter columns
print()
print('Hyperparameter columns found in cv_results:')
param_cols_all = [c for c in cv_results.columns if c.startswith('param_')]
for c in param_cols_all:
    print(f'  {c}')


In [ ]:
# XGBoost hyperparameters analyzed
# Automatically detected from cv_results
PARAM_COLS_CANDIDATES = {
    'param_learning_rate'   : 'learning_rate',
    'param_n_estimators'    : 'n_estimators',
    'param_max_depth'       : 'max_depth',
    'param_min_child_weight': 'min_child_weight',
    'param_subsample'       : 'subsample',
    'param_colsample_bytree': 'colsample_bytree',
    'param_reg_alpha'       : 'reg_alpha',
    'param_reg_lambda'      : 'reg_lambda',
    'param_gamma'           : 'gamma',
}

# Keep only those present in the CSV
PARAM_COLS  = [c for c in PARAM_COLS_CANDIDATES if c in cv_results.columns]
PARAM_NAMES = [PARAM_COLS_CANDIDATES[c] for c in PARAM_COLS]

missing = [c for c in PARAM_COLS_CANDIDATES if c not in cv_results.columns]
if missing:
    print(f'⚠️  Not found (may have a different name): {missing}')

X = cv_results[PARAM_COLS].values
y = cv_results['mean_test_score'].values

print(f'Hyperparameters analyzed ({len(PARAM_NAMES)}): {PARAM_NAMES}')
print(f'Shape X: {X.shape}')


## 3. Hyperparameter Importance

Importance is estimated by training a Random Forest over the 50 trials,
where the hyperparameters are the features and the mean R² (GroupKFold) is the target.
The Spearman correlation indicates the direction of the effect
(positive = higher value → better performance).

**Result:** `max_depth` dominates with **77.6%** of total importance,
followed by `learning_rate` (10.8%) and `n_estimators` (9.3%).
The top-2 account for **88.4%** of performance variation across trials.

| Hyperparameter | Importance (%) | Spearman | Direction |
|---|---|---|---|
| `max_depth` | **77.6** | +0.325 | ↑ higher = better |
| `learning_rate` | 10.8 | -0.158 | ↓ lower = better |
| `n_estimators` | 9.3 | +0.022 | ~ neutral |
| `colsample_bytree` | 0.9 | +0.285 | ↑ higher = better |
| `reg_alpha` | 0.5 | -0.093 | ~ neutral |
| `reg_lambda` | 0.3 | -0.014 | ~ neutral |
| `gamma` | 0.3 | -0.218 | ↓ lower = better |
| `subsample` | 0.2 | -0.018 | ~ neutral |
| `min_child_weight` | 0.1 | -0.220 | ↓ lower = better |


In [ ]:
# Importance via Random Forest
rf = RandomForestRegressor(n_estimators=500, random_state=42)
rf.fit(X, y)
importances = rf.feature_importances_

# Spearman correlation
spearman_corr = [spearmanr(cv_results[col], y).statistic for col in PARAM_COLS]

# Summary table
summary_df = pd.DataFrame({
    'Hyperparameter'  : PARAM_NAMES,
    'Importance (%)'  : (importances * 100).round(1),
    'Spearman'       : [round(s, 3) for s in spearman_corr],
}).sort_values('Importance (%)', ascending=False).reset_index(drop=True)

summary_df['Direction'] = summary_df['Spearman'].apply(
    lambda x: '↓ lower = better' if x < -0.1 else ('↑ higher = better' if x > 0.1 else '~ neutral')
)

print(summary_df.to_string(index=False))

top2_acc  = summary_df['Importance (%)'].head(2).sum()
top1_name = summary_df.iloc[0]['Hyperparameter']
top2_name = summary_df.iloc[1]['Hyperparameter']
top1_imp  = summary_df.iloc[0]['Importance (%)']
top2_imp  = summary_df.iloc[1]['Importance (%)']

print(f'\nTop-2 cumulative: {top2_acc:.1f}%  ({top1_name}: {top1_imp:.1f}% + {top2_name}: {top2_imp:.1f}%)')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('XGBoost — Hyperparameter Importance',
             fontsize=13, fontweight='bold', y=1.01)

order = np.argsort(importances)

# Importance
colors_imp = ['#2196F3' if importances[i] >= 0.10 else '#90CAF9' for i in order]
vals_imp   = [importances[i] * 100 for i in order]
names_ord  = [PARAM_NAMES[i] for i in order]
axes[0].barh(names_ord, vals_imp, color=colors_imp)
for i, val in enumerate(vals_imp):
    axes[0].text(val + 0.4, i, f'{val:.1f}%', va='center', fontsize=9)
axes[0].set_xlabel('Importance (%)')
axes[0].set_title('Importance via Random Forest\n(% of R² variation explained)')
xlim_max = max(vals_imp) * 1.25
axes[0].set_xlim(0, xlim_max)

# Spearman
spear_ord   = [spearman_corr[i] for i in order]
colors_corr = ['#E53935' if v < 0 else '#43A047' for v in spear_ord]
axes[1].barh(names_ord, spear_ord, color=colors_corr)
axes[1].axvline(0, color='black', linewidth=0.8)
for i, val in enumerate(spear_ord):
    offset = 0.01 if val >= 0 else -0.01
    ha     = 'left' if val >= 0 else 'right'
    axes[1].text(val + offset, i, f'{val:.2f}', va='center', ha=ha, fontsize=9)
axes[1].set_xlabel('Spearman Correlation with R²')
axes[1].set_title('Direction of Effect\n(red = higher → worse | green = higher → better)')

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'xgboost_hyperparameter_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')


## 4. Individual Sensitivity

Scatter plots of mean R² as a function of each hyperparameter (top 4:
`max_depth`, `learning_rate`, `n_estimators`, `colsample_bytree`).

The dashed line indicates the linear trend and the vertical dotted line
marks the best trial found by `RandomizedSearchCV`.

> **Note:** `max_depth=4` is the value from the best trial. The positive trend
> (Spearman=+0.33) suggests that greater depths tend to yield better R²,
> but the tested range (typically 3–8) should be taken into account.


In [ ]:
top4_idx = np.argsort(importances)[::-1][:4]
best_idx = cv_results['mean_test_score'].idxmax()

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle(
    'XGBoost — Individual Sensitivity by Hyperparameter\n(Top 4 most important)',
    fontsize=13, fontweight='bold'
)

for ax, idx in zip(axes.flatten(), top4_idx):
    x_vals = cv_results[PARAM_COLS[idx]].values
    sc = ax.scatter(x_vals, y, c=y, cmap='RdYlGn', alpha=0.8,
                    edgecolors='k', linewidth=0.3, s=65, zorder=3)

    # Linear trend
    z      = np.polyfit(x_vals, y, 1)
    x_line = np.linspace(x_vals.min(), x_vals.max(), 100)
    ax.plot(x_line, np.poly1d(z)(x_line), 'k--', alpha=0.5, linewidth=1.2, label='Trend')

    # Melhor trial
    best_val = cv_results.loc[best_idx, PARAM_COLS[idx]]
    ax.axvline(best_val, color='#1565C0', linestyle=':', linewidth=1.8,
               alpha=0.85, label=f'Best trial ({best_val:.3g})')

    spear = spearman_corr[idx]
    imp   = importances[idx] * 100
    ax.set_xlabel(PARAM_NAMES[idx], fontsize=11)
    ax.set_ylabel('Mean R² (GroupKFold)', fontsize=10)
    ax.set_title(f'{PARAM_NAMES[idx]}\nImportance: {imp:.1f}%  |  Spearman: {spear:.2f}', fontsize=10)
    ax.legend(fontsize=8, loc='best')
    plt.colorbar(sc, ax=ax, label='R²')

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'xgboost_sensitivity_individual.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')


## 5. Interaction Analysis

The two most important hyperparameters in XGBoost are `max_depth` and `learning_rate`,
which together control the model's capacity and learning pace.

- **`max_depth`** controls the complexity of each tree — larger values allow
  capturing more complex relationships, but increase the risk of overfitting.
- **`learning_rate`** controls the update step — smaller values typically
  require more estimators (`n_estimators`) to converge, a classic
  trade-off in gradient boosting.

The R²-colored scatter reveals the regions of the hyperparameter space
associated with the best performance, with the star (★) marking the best trial.


In [ ]:
# Dynamic names of top-2 for titles
top2_cols = [PARAM_COLS[i]  for i in np.argsort(importances)[::-1][:2]]
top2_nms  = [PARAM_NAMES[i] for i in np.argsort(importances)[::-1][:2]]
top2_imps = sorted(importances, reverse=True)[:2]

# Third most important hyperparameter
top3_col = PARAM_COLS[np.argsort(importances)[::-1][2]]
top3_nm  = PARAM_NAMES[np.argsort(importances)[::-1][2]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle('XGBoost — Interação entre Hiperparâmetros',
             fontsize=13, fontweight='bold')

best_idx = cv_results['mean_test_score'].idxmax()

# Top-1 × Top-2
ax = axes[0]
sc = ax.scatter(cv_results[top2_cols[0]], cv_results[top2_cols[1]],
                c=cv_results['mean_test_score'], cmap='RdYlGn',
                s=85, alpha=0.85, edgecolors='k', linewidth=0.3)
plt.colorbar(sc, ax=ax, label='Mean R²')
ax.scatter(cv_results.loc[best_idx, top2_cols[0]],
           cv_results.loc[best_idx, top2_cols[1]],
           marker='*', s=350, color='#1565C0', zorder=5, label='Best trial')
ax.set_xlabel(top2_nms[0], fontsize=11)
ax.set_ylabel(top2_nms[1], fontsize=11)
ax.set_title(
    f'{top2_nms[0]} × {top2_nms[1]}\n'
    f'({top2_imps[0]*100:.1f}% + {top2_imps[1]*100:.1f}% = {sum(top2_imps)*100:.1f}% of total variation)',
    fontsize=10
)
ax.legend(fontsize=9)

# Top-1 × Top-3
ax = axes[1]
sc = ax.scatter(cv_results[top2_cols[0]], cv_results[top3_col],
                c=cv_results['mean_test_score'], cmap='RdYlGn',
                s=85, alpha=0.85, edgecolors='k', linewidth=0.3)
plt.colorbar(sc, ax=ax, label='Mean R²')
ax.scatter(cv_results.loc[best_idx, top2_cols[0]],
           cv_results.loc[best_idx, top3_col],
           marker='*', s=350, color='#1565C0', zorder=5, label='Best trial')
ax.set_xlabel(top2_nms[0], fontsize=11)
ax.set_ylabel(top3_nm, fontsize=11)
ax.set_title(f'{top2_nms[0]} × {top3_nm}', fontsize=10)
ax.legend(fontsize=9)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'xgboost_sensitivity_interaction.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')


## 6. Model Stability

A desirable characteristic in models for geophysical applications is robustness
to the choice of hyperparameters. The distribution of R² values across the 50 trials and the
ranking reveal how sensitive the model is to configuration variations.

> **Result:** the total range is **0.0671** (best: 0.8900, worst: 0.8228),
> with a mean of 0.8733 ± 0.0190. **Moderate** stability — `max_depth` has
> dominant impact (77.6%), but even the worst trial still yields R² > 0.82,
> indicating that XGBoost is robust enough for operational use.


In [ ]:
scores_sorted = np.sort(cv_results['mean_test_score'].values)[::-1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('XGBoost — Stability Across Trials',
             fontsize=13, fontweight='bold')

# Distribution of R²
ax = axes[0]
ax.hist(cv_results['mean_test_score'], bins=15, color='#2196F3',
        edgecolor='white', alpha=0.85)
ax.axvline(scores_sorted[0], color='#1B5E20', linestyle='--', linewidth=1.8,
           label=f'Best: {scores_sorted[0]:.4f}')
ax.axvline(scores_sorted.mean(), color='#E65100', linestyle=':', linewidth=1.8,
           label=f'Mean: {scores_sorted.mean():.4f}')
ax.set_xlabel('Mean R² (GroupKFold)')
ax.set_ylabel('Frequency')
ax.set_title('R² Distribution Across 50 Trials')
ax.legend(fontsize=9)

# Ranking of trials
ax = axes[1]
ax.plot(range(1, len(scores_sorted) + 1), scores_sorted,
        'o-', color='#2196F3', markersize=5, linewidth=1.2, alpha=0.8)
ax.fill_between(range(1, len(scores_sorted) + 1), scores_sorted,
                scores_sorted.min(), alpha=0.15, color='#2196F3')
ax.axhline(scores_sorted[0], color='#1B5E20', linestyle='--', linewidth=1.5,
           label=f'Best: {scores_sorted[0]:.4f}')
ax.axhline(scores_sorted.mean(), color='#E65100', linestyle=':', linewidth=1.5,
           label=f'Mean: {scores_sorted.mean():.4f}')
ax.set_xlabel('Trial Rank (1 = best)')
ax.set_ylabel('Mean R² (GroupKFold)')
ax.set_title(
    f'Performance by Rank\nTotal range: {scores_sorted[0] - scores_sorted[-1]:.4f}'
)
ax.legend(fontsize=9)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'xgboost_stability.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')


## 7. Summary and Conclusions

**Main findings from the XGBoost sensitivity analysis:**

1. **`max_depth` is the dominant parameter (77.6%)** — higher values tend to improve
   performance (Spearman=+0.33). The best trial used `max_depth=4`.

2. **`learning_rate` and `n_estimators`** contribute 10.8% and 9.3%, respectively.
   A lower `learning_rate` tends to be better (Spearman=-0.16), consistent with
   the gradient boosting literature.

3. **The remaining 6 hyperparameters** (`colsample_bytree`, `reg_alpha`, `reg_lambda`,
   `gamma`, `subsample`, `min_child_weight`) contribute only **11.6%** combined
   — their tuning has marginal impact on performance.

4. **Moderate stability:** a range of 0.0671 across the 50 trials. `max_depth`
   has significant impact, but even suboptimal configurations yield
   R² > 0.82 — the model is robust for operational use.

5. **Practical implication:** in future applications in the Sergipe-Alagoas Basin,
   prioritizing the tuning of `max_depth` and `learning_rate` is sufficient to
   achieve near-optimal performance.


In [ ]:
amplitude  = scores_sorted[0] - scores_sorted[-1]
top1_spear = summary_df.iloc[0]['Spearman']
top2_spear = summary_df.iloc[1]['Spearman']
top1_dir   = 'lower' if top1_spear < 0 else 'higher'
top2_dir   = 'lower' if top2_spear < 0 else 'higher'

print('=' * 65)
print('SENSITIVITY ANALYSIS — XGBoost')
print('Sergipe-Alagoas Basin | DT Prediction | LOWO (28 wells)')
print('=' * 65)

print(f'\nTrials evaluated : {len(cv_results)}')
print(f'Best trial R²    : {scores_sorted[0]:.4f}')
print(f'Worst trial R²   : {scores_sorted[-1]:.4f}')
print(f'Total range      : {amplitude:.4f}')

print(f'\n{"-" * 65}')
print('HYPERPARAMETER IMPORTANCE')
print(f'{"-" * 65}')
for _, row in summary_df.iterrows():
    bar = '█' * int(row['Importance (%)'] / 2)
    print(f'  {row["Hyperparameter"]:25s}: {row["Importance (%)"] :5.1f}%  {bar}')

print(f'\n{"-" * 65}')
print('CONCLUSIONS')
print(f'{"-" * 65}')
print(f'''
1. DOMINANCE OF TWO PARAMETERS
   {top1_name} ({top1_imp:.1f}%) and {top2_name} ({top2_imp:.1f}%) explain
   {top2_acc:.1f}% of performance variation across the {len(cv_results)} trials.
   The remaining hyperparameters contribute only {100 - top2_acc:.1f}% combined.

2. DIRECTION OF EFFECTS
   - {top1_name} {top1_dir} → better performance  (Spearman = {top1_spear:.2f})
   - {top2_name} {top2_dir} → better performance  (Spearman = {top2_spear:.2f})

3. MODEL STABILITY
   The total R² range across the {len(cv_results)} trials is {amplitude:.4f}.
   {'High stability: the model is robust to the choice of hyperparameters.' if amplitude < 0.02 else 'Moderate stability: hyperparameters have a relevant impact on performance.'}

4. PRACTICAL IMPLICATION
   Controlling {top1_name} is the most critical factor
   for XGBoost performance in the Sergipe-Alagoas Basin.
''')
print('=' * 65)
